# Lecture 6: Classification Algorithms
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Implement Decision Tree, Naive Bayes, KNN, Random Forest, and Gradient Boosting
2. Visualise a decision tree
3. Compare algorithms on the same dataset
4. Interpret feature importance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import warnings; warnings.filterwarnings('ignore')
print('Ready.')

## 6.1 Loading the Dataset

We use the Breast Cancer Wisconsin dataset — a classic binary classification problem.
- **Target:** Malignant (1) vs Benign (0)
- **Features:** 30 numerical features derived from tumour measurements

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame
df['target'] = data.target
print(f'Shape: {df.shape}')
print(f'\nClass distribution:')
print(df['target'].value_counts().rename({0: 'Malignant', 1: 'Benign'}))

In [ ]:
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

## 6.2 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
print(f'Decision Tree Accuracy: {accuracy_score(y_test, dt.predict(X_test)):.4f}')

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(dt, feature_names=data.feature_names, class_names=data.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=4)')
plt.tight_layout()
plt.show()

## 6.3 Naive Bayes

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
print(f'Naive Bayes Accuracy: {accuracy_score(y_test, gnb.predict(X_test)):.4f}')

## 6.4 K-Nearest Neighbours

In [ ]:
# Find optimal k
k_results = []
for k in range(1, 26):
    knn = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn, X_tr_sc, y_train, cv=5).mean()
    k_results.append({'k': k, 'cv_accuracy': score})

kr = pd.DataFrame(k_results)
best_k = kr.loc[kr.cv_accuracy.idxmax(), 'k']
print(f'Best k: {best_k} (CV accuracy: {kr.cv_accuracy.max():.4f})')

plt.figure(figsize=(9, 4))
plt.plot(kr.k, kr.cv_accuracy, 'b-o', markersize=5)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k')
plt.ylabel('CV Accuracy')
plt.title('KNN: Finding Optimal k')
plt.legend()
plt.tight_layout()
plt.show()

knn_best = KNeighborsClassifier(n_neighbors=int(best_k))
knn_best.fit(X_tr_sc, y_train)
print(f'KNN Test Accuracy (k={best_k}): {accuracy_score(y_test, knn_best.predict(X_te_sc)):.4f}')

## 6.5 Random Forest and Gradient Boosting

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f'Random Forest Accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}')

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                 max_depth=3, random_state=42)
gb.fit(X_train, y_train)
print(f'Gradient Boosting Accuracy: {accuracy_score(y_test, gb.predict(X_test)):.4f}')

## 6.6 Algorithm Comparison

In [ ]:
models = {
    'Decision Tree (d=4)': dt.predict(X_test),
    'Naive Bayes':         gnb.predict(X_test),
    f'KNN (k={best_k})':  knn_best.predict(X_te_sc),
    'Random Forest':       rf.predict(X_test),
    'Gradient Boosting':   gb.predict(X_test),
}
results = [{'Algorithm': name, 'Test Accuracy': accuracy_score(y_test, preds)}
           for name, preds in models.items()]
results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print(results_df.to_string(index=False))

plt.figure(figsize=(9, 4))
plt.barh(results_df['Algorithm'], results_df['Test Accuracy'], color='steelblue')
plt.xlabel('Test Accuracy')
plt.title('Classification Algorithm Comparison — Breast Cancer Dataset')
plt.xlim(0.85, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance from Random Forest
fi = pd.DataFrame({'feature': data.feature_names,
                   'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(9, 5))
plt.barh(fi['feature'][::-1], fi['importance'][::-1], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Top 10 Features — Random Forest')
plt.tight_layout()
plt.show()

### Exercise

Using the Breast Cancer dataset:

1. Print the full classification_report for the Random Forest model.
2. Tune the Random Forest by trying n_estimators in [50, 100, 200] and max_depth in [None, 5, 10] using cross-validation. Report the best combination.
3. Compare the classification report of the best Random Forest vs Gradient Boosting. Which has better Recall for the malignant class, and why does Recall matter more than Precision in this medical context?

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*